# Gemini-Logical MSD Reproduction with Overrotated Input Preparation

This notebook reproduces the logical MSD demo using the Gemini logical simulator, but models input-state infidelity by **overrotating the logical `u3` preparation** instead of injecting explicit noise instructions into a custom logical initializer.

Key differences from [`/Users/jasonhan/Desktop/qmain/kirin-workspace/bloqade-lanes/demo/magic_state_distillation_reprod.ipynb`](/Users/jasonhan/Desktop/qmain/kirin-workspace/bloqade-lanes/demo/magic_state_distillation_reprod.ipynb):
- no custom `build_task(...)` wrapper
- tasks are created directly with `GeminiLogicalSimulator.task(...)`
- no manual noise instructions in the MSD input-prep kernel
- all analysis below uses the **full logical post-processing** path (`run_detectors=False`)


In [1]:
import math
import sys
from pathlib import Path

import numpy as np

PROJECT_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
for candidate in PROJECT_ROOT_CANDIDATES:
    candidate = candidate.resolve()
    if (candidate / 'demo' / 'msd_utils').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError('Could not locate repo root containing demo/msd_utils.')

from bloqade.lanes import GeminiLogicalSimulator
from bloqade.lanes.steane_defaults import steane7_m2dets, steane7_m2obs

from demo.msd_utils.core import (
    BasisDataset,
    DEFAULT_BASIS_LABELS,
    DEFAULT_IDEAL_FACTORY_ACCEPTANCE,
    DEFAULT_TARGET_BLOCH,
    fidelity_from_counts,
    logical_expectation,
)
from demo.msd_utils.circuits import build_naive_kernel_bundle


In [2]:
IDEAL_THETA = 0.3041 * math.pi
PHI = 0.25 * math.pi
LAM = 0.0

TARGET_INPUT_FIDELITY = 0.97
POSTERIOR_SAMPLES = 20_000
CALIBRATION_SHOTS = 20_000
SUMMARY_SHOTS = 200_000
CHUNK_SIZE = 10_000

BASIS_LABELS = DEFAULT_BASIS_LABELS
IDEAL_FACTORY_ACCEPTANCE = DEFAULT_IDEAL_FACTORY_ACCEPTANCE
TARGET_BLOCH = DEFAULT_TARGET_BLOCH

M2DETS_5 = steane7_m2dets(5)
M2OBS_5 = steane7_m2obs(5)
M2DETS_1 = steane7_m2dets(1)
M2OBS_1 = steane7_m2obs(1)


For a pure-state overrotation proxy with fixed `phi`, the fidelity to the ideal target changes as

\[
F = \cos^2(\delta	heta/2).
\]

So for a desired input fidelity `F`, we can choose

\[
\delta	heta = 2rccos(\sqrt{F}).
\]


In [3]:
def theta_offset_for_fidelity(target_fidelity: float) -> float:
    target_fidelity = float(np.clip(target_fidelity, 0.0, 1.0))
    return 2.0 * math.acos(math.sqrt(target_fidelity))


def bloch_from_u3(theta: float, phi: float) -> np.ndarray:
    return np.array(
        [
            math.sin(theta) * math.cos(phi),
            math.sin(theta) * math.sin(phi),
            math.cos(theta),
        ],
        dtype=np.float64,
    )


PREP_DELTA_THETA = theta_offset_for_fidelity(TARGET_INPUT_FIDELITY)
PREP_THETA = IDEAL_THETA + PREP_DELTA_THETA
PREP_BLOCH = bloch_from_u3(PREP_THETA, PHI)
PREP_FIDELITY = 0.5 + float(np.dot(PREP_BLOCH, TARGET_BLOCH)) / 2.0

print('ideal theta / pi        =', IDEAL_THETA / math.pi)
print('prep theta / pi         =', PREP_THETA / math.pi)
print('delta theta / pi        =', PREP_DELTA_THETA / math.pi)
print('analytic prep bloch     =', tuple(float(x) for x in PREP_BLOCH))
print('analytic prep fidelity  =', PREP_FIDELITY)


ideal theta / pi        = 0.3041
prep theta / pi         = 0.41492468660445936
delta theta / pi        = 0.11082468660445939
analytic prep bloch     = (0.6820009252488507, 0.6820009252488506, 0.2641012607304692)
analytic prep fidelity  = 0.9699928847680258


In [4]:
kernel_bundle = build_naive_kernel_bundle(PREP_THETA, PHI, LAM, output_qubit=0)
DISTILLED_KERNELS = kernel_bundle.distilled
INJECTED_KERNELS = kernel_bundle.injected

kernel_bundle


NaiveKernelBundle(distilled={'X': Method("distilled_x"), 'Y': Method("distilled_y"), 'Z': Method("distilled_z")}, injected={'X': Method("injected_x"), 'Y': Method("injected_y"), 'Z': Method("injected_z")})

In [5]:
sim = GeminiLogicalSimulator()

distilled_tasks = {
    basis: sim.task(DISTILLED_KERNELS[basis], m2dets=M2DETS_5, m2obs=M2OBS_5)
    for basis in BASIS_LABELS
}
injected_tasks = {
    basis: sim.task(INJECTED_KERNELS[basis], m2dets=M2DETS_1, m2obs=M2OBS_1)
    for basis in BASIS_LABELS
}

distilled_tasks


{'X': GeminiLogicalSimulatorTask(logical_squin_kernel=Method("distilled_x"), noise_model=SimpleNoiseModel(lane_noise=Method("lane_noise"), idle_noise=Method("idle_noise"), cz_unpaired_noise=Method("cz_unpaired_noise"), cz_paired_noise=Method("cz_paired_noise"), global_rz_noise=Method("global_rz_noise"), local_rz_noise=Method("local_rz_noise"), global_r_noise=Method("global_r_noise"), local_r_noise=Method("local_r_noise")), _thread_pool_executor=<concurrent.futures.thread.ThreadPoolExecutor object at 0x120b1a000>),
 'Y': GeminiLogicalSimulatorTask(logical_squin_kernel=Method("distilled_y"), noise_model=SimpleNoiseModel(lane_noise=Method("lane_noise"), idle_noise=Method("idle_noise"), cz_unpaired_noise=Method("cz_unpaired_noise"), cz_paired_noise=Method("cz_paired_noise"), global_rz_noise=Method("global_rz_noise"), local_rz_noise=Method("local_rz_noise"), global_r_noise=Method("global_r_noise"), local_r_noise=Method("local_r_noise")), _thread_pool_executor=<concurrent.futures.thread.Thre

The helper below intentionally uses `run_detectors=False`, so it samples raw measurements and then applies the task's full logical post-processing. That avoids the detector-sampler mismatch we saw in the other notebook while debugging noiseless logical observables.


In [6]:
def run_task_full(
    task,
    shots: int,
    *,
    with_noise: bool = True,
    chunk_size: int | None = None,
) -> BasisDataset:
    if chunk_size is None or shots <= chunk_size:
        result = task.run(shots, with_noise=with_noise, run_detectors=False)
        return BasisDataset(
            detectors=np.asarray(result.detectors, dtype=np.uint8),
            observables=np.asarray(result.observables, dtype=np.uint8),
        )

    det_chunks = []
    obs_chunks = []
    remaining = shots
    while remaining > 0:
        batch = min(chunk_size, remaining)
        result = task.run(batch, with_noise=with_noise, run_detectors=False)
        det_chunks.append(np.asarray(result.detectors, dtype=np.uint8))
        obs_chunks.append(np.asarray(result.observables, dtype=np.uint8))
        remaining -= batch

    return BasisDataset(
        detectors=np.concatenate(det_chunks, axis=0),
        observables=np.concatenate(obs_chunks, axis=0),
    )


def best_sign_vector(raw_bloch: np.ndarray, target_bloch: np.ndarray = TARGET_BLOCH) -> np.ndarray:
    sign_candidates = [
        np.array([sx, sy, sz], dtype=np.float64)
        for sx in (-1.0, 1.0)
        for sy in (-1.0, 1.0)
        for sz in (-1.0, 1.0)
    ]
    return max(sign_candidates, key=lambda sign: float(np.dot(raw_bloch * sign, target_bloch)))


def infer_factory_target_full(
    task_map,
    *,
    shots: int = CALIBRATION_SHOTS,
    basis_labels=BASIS_LABELS,
    ideal_factory_acceptance: float = IDEAL_FACTORY_ACCEPTANCE,
) -> np.ndarray:
    counts = {}
    total = 0
    for basis in basis_labels:
        data = run_task_full(task_map[basis], shots, with_noise=False, chunk_size=CHUNK_SIZE)
        for row in data.observables[:, 1:]:
            key = tuple(int(x) for x in row)
            counts[key] = counts.get(key, 0) + 1
            total += 1

    ranked = sorted(
        counts.items(),
        key=lambda item: (abs(item[1] / total - ideal_factory_acceptance), -item[1]),
    )
    print('Top noiseless ancilla branches:')
    for pattern, count in ranked[:8]:
        print(pattern, count / total)
    return np.asarray(ranked[0][0], dtype=np.uint8)


def infer_injected_sign_vector_full(task_map, *, shots: int = CALIBRATION_SHOTS) -> np.ndarray:
    raw_bloch = []
    for basis in BASIS_LABELS:
        data = run_task_full(task_map[basis], shots, with_noise=False, chunk_size=CHUNK_SIZE)
        raw_bloch.append(logical_expectation(data.observables[:, 0]))
    raw_bloch = np.asarray(raw_bloch, dtype=np.float64)
    sign = best_sign_vector(raw_bloch, TARGET_BLOCH)
    print('Noiseless injected Bloch:', raw_bloch)
    print('Chosen injected sign vector:', sign)
    return sign


def infer_distilled_sign_vector_full(
    task_map,
    factory_target: np.ndarray,
    *,
    shots: int = CALIBRATION_SHOTS,
) -> np.ndarray:
    raw_bloch = []
    for basis in BASIS_LABELS:
        data = run_task_full(task_map[basis], shots, with_noise=False, chunk_size=CHUNK_SIZE)
        mask = np.all(data.observables[:, 1:] == factory_target, axis=1)
        raw_bloch.append(logical_expectation(data.observables[mask, 0]))
    raw_bloch = np.asarray(raw_bloch, dtype=np.float64)
    sign = best_sign_vector(raw_bloch, TARGET_BLOCH)
    print('Noiseless distilled accepted-branch Bloch:', raw_bloch)
    print('Chosen distilled sign vector:', sign)
    return sign


def summarize_injected(
    task_map,
    *,
    shots: int,
    with_noise: bool,
    posterior_samples: int,
    sign_vector,
    require_zero_detectors: bool = False,
):
    corrected = {}
    accepted_fraction_by_basis = {}
    for basis in BASIS_LABELS:
        data = run_task_full(task_map[basis], shots, with_noise=with_noise, chunk_size=CHUNK_SIZE)
        mask = np.ones(len(data.observables), dtype=bool)
        if require_zero_detectors:
            mask &= np.all(data.detectors == 0, axis=1)
        corrected[basis] = data.observables[mask, 0].astype(np.uint8)
        accepted_fraction_by_basis[basis] = float(np.mean(mask))

    summary = fidelity_from_counts(
        corrected['X'],
        corrected['Y'],
        corrected['Z'],
        posterior_samples,
        sign_vector=sign_vector,
        target_bloch=TARGET_BLOCH,
    )
    summary['accepted_fraction'] = float(np.mean(list(accepted_fraction_by_basis.values())))
    summary['accepted_fraction_by_basis'] = accepted_fraction_by_basis
    return summary


def summarize_distilled(
    task_map,
    factory_target: np.ndarray,
    *,
    shots: int,
    with_noise: bool,
    posterior_samples: int,
    sign_vector,
    require_zero_detectors: bool = True,
):
    corrected = {}
    accepted_fraction_by_basis = {}
    total_kept = 0
    total_shots = 0

    for basis in BASIS_LABELS:
        data = run_task_full(task_map[basis], shots, with_noise=with_noise, chunk_size=CHUNK_SIZE)
        mask = np.all(data.observables[:, 1:] == factory_target, axis=1)
        if require_zero_detectors:
            mask &= np.all(data.detectors == 0, axis=1)
        corrected[basis] = data.observables[mask, 0].astype(np.uint8)
        accepted_fraction_by_basis[basis] = float(np.mean(mask))
        total_kept += int(mask.sum())
        total_shots += len(mask)

    summary = fidelity_from_counts(
        corrected['X'],
        corrected['Y'],
        corrected['Z'],
        posterior_samples,
        sign_vector=sign_vector,
        target_bloch=TARGET_BLOCH,
    )
    summary['accepted_fraction'] = total_kept / total_shots
    summary['accepted_fraction_by_basis'] = accepted_fraction_by_basis
    summary['factory_target'] = tuple(int(x) for x in factory_target.tolist())
    return summary


In [7]:
FACTORY_TARGET = infer_factory_target_full(distilled_tasks)
INJECTED_SIGN_VECTOR = infer_injected_sign_vector_full(injected_tasks)
DISTILLED_SIGN_VECTOR = infer_distilled_sign_vector_full(distilled_tasks, FACTORY_TARGET)

print('FACTORY_TARGET =', FACTORY_TARGET)


Top noiseless ancilla branches:
(0, 0, 0, 0) 0.14275
(0, 1, 1, 0) 0.07816666666666666
(1, 1, 1, 0) 0.07293333333333334
(1, 0, 0, 0) 0.0729
(0, 0, 1, 1) 0.07288333333333333
(1, 1, 1, 1) 0.07203333333333334
(1, 0, 0, 1) 0.07171666666666666
(0, 1, 0, 1) 0.07046666666666666
Noiseless injected Bloch: [ 0.6825 -0.6811  0.2689]
Chosen injected sign vector: [ 1. -1.  1.]
Noiseless distilled accepted-branch Bloch: [ 0.50086655 -0.58897418  0.64476386]
Chosen distilled sign vector: [ 1. -1.  1.]
FACTORY_TARGET = [0 0 0 0]


In [8]:
injected_noiseless = summarize_injected(
    injected_tasks,
    shots=SUMMARY_SHOTS,
    with_noise=False,
    posterior_samples=POSTERIOR_SAMPLES,
    sign_vector=INJECTED_SIGN_VECTOR,
)
distilled_noiseless = summarize_distilled(
    distilled_tasks,
    FACTORY_TARGET,
    shots=SUMMARY_SHOTS,
    with_noise=False,
    posterior_samples=POSTERIOR_SAMPLES,
    sign_vector=DISTILLED_SIGN_VECTOR,
    require_zero_detectors=True,
)

print('Injected noiseless summary:')
injected_noiseless


/Users/jasonhan/Desktop/qmain/kirin-workspace/bloqade-lanes/demo/msd_utils/core.py:102: RuntimeWarning: divide by zero encountered in matmul
  fidelities = 0.5 + (corrected_points @ target_bloch) / 2.0
/Users/jasonhan/Desktop/qmain/kirin-workspace/bloqade-lanes/demo/msd_utils/core.py:102: RuntimeWarning: overflow encountered in matmul
  fidelities = 0.5 + (corrected_points @ target_bloch) / 2.0
/Users/jasonhan/Desktop/qmain/kirin-workspace/bloqade-lanes/demo/msd_utils/core.py:102: RuntimeWarning: invalid value encountered in matmul
  fidelities = 0.5 + (corrected_points @ target_bloch) / 2.0


Injected noiseless summary:


{'point': 0.9706790334541506,
 'median': 0.9667616230351851,
 'low': 0.9666673243469157,
 'high': 0.9668559217234545,
 'bloch': (0.68094, 0.68513, 0.26441),
 'accepted_fraction': 1.0,
 'accepted_fraction_by_basis': {'X': 1.0, 'Y': 1.0, 'Z': 1.0}}

## Ideal Injected Reference

This section rebuilds the single-qubit injected kernels with the **ideal** preparation angle and reports the noiseless injected-state summary. It gives a direct reference for the fidelity you would get from perfect logical magic-state preparation before any overrotation proxy is applied.


In [9]:
ideal_kernel_bundle = build_naive_kernel_bundle(IDEAL_THETA, PHI, LAM, output_qubit=0)
ideal_injected_tasks = {
    basis: sim.task(ideal_kernel_bundle.injected[basis], m2dets=M2DETS_1, m2obs=M2OBS_1)
    for basis in BASIS_LABELS
}
IDEAL_INJECTED_SIGN_VECTOR = infer_injected_sign_vector_full(ideal_injected_tasks)
ideal_injected_noiseless = summarize_injected(
    ideal_injected_tasks,
    shots=SUMMARY_SHOTS,
    with_noise=False,
    posterior_samples=POSTERIOR_SAMPLES,
    sign_vector=IDEAL_INJECTED_SIGN_VECTOR,
)

print('Ideal injected noiseless summary:')
ideal_injected_noiseless


Noiseless injected Bloch: [ 0.5762 -0.5736  0.5745]
Chosen injected sign vector: [ 1. -1.  1.]
Ideal injected noiseless summary:


{'point': 0.9984582416562094,
 'median': 0.9825454897380235,
 'low': 0.9820321321399476,
 'high': 0.9830588473360993,
 'bloch': (0.57745, 0.57628, 0.57298),
 'accepted_fraction': 1.0,
 'accepted_fraction_by_basis': {'X': 1.0, 'Y': 1.0, 'Z': 1.0}}

In [10]:
print('Distilled noiseless summary:')
distilled_noiseless


Distilled noiseless summary:


{'point': 0.9967496333534607,
 'median': 0.994860521716882,
 'low': 0.9941342155342672,
 'high': 0.9955868278994968,
 'bloch': (0.4979936349799364, 0.5582796447849556, 0.6645179274539182),
 'accepted_fraction': 0.14387,
 'accepted_fraction_by_basis': {'X': 0.14454, 'Y': 0.143575, 'Z': 0.143495},
 'factory_target': (0, 0, 0, 0)}

If you want to include the simulator's default physical noise model on top of the overrotated input preparation, rerun the same summaries with `with_noise=True`.


## Noiseless Distillation Expectations

These cells estimate the raw and post-selected noiseless logical expectations of the distilled output qubit in the `X`, `Y`, and `Z` tomography bases, using the full logical post-processing path.


In [11]:
NOISELESS_EXPECTATION_SHOTS = 1_000_000

x_data = run_task_full(
    distilled_tasks['X'],
    NOISELESS_EXPECTATION_SHOTS,
    with_noise=False,
    chunk_size=CHUNK_SIZE,
)
y_data = run_task_full(
    distilled_tasks['Y'],
    NOISELESS_EXPECTATION_SHOTS,
    with_noise=False,
    chunk_size=CHUNK_SIZE,
)
z_data = run_task_full(
    distilled_tasks['Z'],
    NOISELESS_EXPECTATION_SHOTS,
    with_noise=False,
    chunk_size=CHUNK_SIZE,
)

ex = logical_expectation(x_data.observables[:, 0])
ey = logical_expectation(y_data.observables[:, 0])
ez = logical_expectation(z_data.observables[:, 0])

print('Noiseless <X>, <Y>, <Z> =', (ex, ey, ez))


Noiseless <X>, <Y>, <Z> = (0.12374, -0.04781, 0.318764)


In [12]:
FACTORY_TARGET_NOISELESS = np.array(FACTORY_TARGET, dtype=np.uint8)

def accepted_output_bits(data, factory_target):
    perfect_stabilizers = np.all(data.detectors == 0, axis=1)
    correct_syndrome = np.all(data.observables[:, 1:] == factory_target, axis=1)
    mask = perfect_stabilizers & correct_syndrome
    return {
        'bits': data.observables[mask, 0].astype(np.uint8),
        'rate_total': float(mask.mean()),
        'rate_stabilizers': float(perfect_stabilizers.mean()),
        'rate_syndrome_given_stabilizers': (
            float(mask.sum() / perfect_stabilizers.sum())
            if perfect_stabilizers.sum() > 0
            else float('nan')
        ),
    }

x = accepted_output_bits(x_data, FACTORY_TARGET_NOISELESS)
y = accepted_output_bits(y_data, FACTORY_TARGET_NOISELESS)
z = accepted_output_bits(z_data, FACTORY_TARGET_NOISELESS)

print(
    'Accepted noiseless <X>, <Y>, <Z> =',
    (
        logical_expectation(x['bits']),
        logical_expectation(y['bits']),
        logical_expectation(z['bits']),
    ),
)

print('Total postselection rates =', (x['rate_total'], y['rate_total'], z['rate_total']))
print(
    'Perfect stabilizer rates =',
    (x['rate_stabilizers'], y['rate_stabilizers'], z['rate_stabilizers']),
)
print(
    'Syndrome rates given perfect stabilizers =',
    (
        x['rate_syndrome_given_stabilizers'],
        y['rate_syndrome_given_stabilizers'],
        z['rate_syndrome_given_stabilizers'],
    ),
)


Accepted noiseless <X>, <Y>, <Z> = (0.5052939130434783, -0.5584683173347997, 0.6636228115791298)
Total postselection rates = (0.14375, 0.143659, 0.144225)
Perfect stabilizer rates = (1.0, 0.999999, 1.0)
Syndrome rates given perfect stabilizers = (0.14375, 0.14365914365914365, 0.144225)


In [13]:
injected_noisy = summarize_injected(
    injected_tasks,
    shots=CALIBRATION_SHOTS,
    with_noise=True,
    posterior_samples=POSTERIOR_SAMPLES,
    sign_vector=INJECTED_SIGN_VECTOR,
)
distilled_noisy = summarize_distilled(
    distilled_tasks,
    FACTORY_TARGET,
    shots=CALIBRATION_SHOTS,
    with_noise=True,
    posterior_samples=POSTERIOR_SAMPLES,
    sign_vector=DISTILLED_SIGN_VECTOR,
    require_zero_detectors=True,
)

print('Injected noisy summary:')
injected_noisy


Injected noisy summary:


{'point': 0.9733406181951147,
 'median': 0.96676161859599,
 'low': 0.9666673168890678,
 'high': 0.9668559203029121,
 'bloch': (0.6843, 0.6818, 0.2736),
 'accepted_fraction': 1.0,
 'accepted_fraction_by_basis': {'X': 1.0, 'Y': 1.0, 'Z': 1.0}}

In [14]:
print('Distilled noisy summary:')
distilled_noisy


Distilled noisy summary:


{'point': 1.0059504033984261,
 'median': 0.985820192124505,
 'low': 0.9849795596196145,
 'high': 0.9949254281025228,
 'bloch': (0.47195357833655704, 0.603290676416819, 0.6774193548387096),
 'accepted_fraction': 0.053033333333333335,
 'accepted_fraction_by_basis': {'X': 0.0517, 'Y': 0.0547, 'Z': 0.0527},
 'factory_target': (0, 0, 0, 0)}